## The Anatomy of a Triton Kernel

In [1]:
import triton
import triton.language as tl
import torch

@triton.jit
def add_kernel(
    x_ptr,          # pointer to input tensor x
    y_ptr,          # pointer to input tensor y
    output_ptr,     # pointer to output tensor
    n_elements,     # total number of elements
    BLOCK_SIZE: tl.constexpr,  # compile-time constant
):
    # 1. Figure out which block (program instance) we are
    pid = tl.program_id(axis=0)

    # 2. Compute the range of elements this block handles
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    # 3. Create a mask for boundary handling
    mask = offsets < n_elements

    # 4. Load data from HBM into SRAM (registers)
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    # 5. Compute
    output = x + y

    # 6. Store results back to HBM
    tl.store(output_ptr + offsets, output, mask=mask)

- `@triton.jit` — Decorator that compiles this function into a GPU kernel. The function body runs on GPU, not CPU.
- `tl.constexpr` — Marks a parameter as a compile-time constant. Triton generates different compiled kernels for different values of BLOCK_SIZE. This enables the compiler to fully unroll loops and optimize memory access patterns.
- `tl.program_id(axis=0)` — Returns which block instance we are in the launch grid. Think of it like blockIdx.x in CUDA, but Triton's abstraction is a "program" rather than a "thread."
- `tl.arange(0, BLOCK_SIZE)` — Creates a vector `[0, 1, 2, ..., BLOCK_SIZE-1]`. Every operation in Triton is implicitly vectorized — you operate on blocks of data, never individual elements.
- `mask` — Handles the tail case. If `n_elements=1000` and `BLOCK_SIZE=256`, the last block would try to access indices `768-1023`, but only `768-999` are valid. The mask prevents out-of-bounds access.
- `tl.load / tl.store` — Move data between HBM and registers. The pointer arithmetic `(x_ptr + offsets)` is how you tell Triton which memory addresses to access. This is the #1 thing that trips up beginners — pointers, not tensor indexing.

## The Launcher (CPU-Side Code)

In [2]:
def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    assert x.is_cuda and y.is_cuda and output.is_cuda
    n_elements = output.numel()

    # Launch grid: how many program instances to create
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    # Launch the kernel
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

# Test
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')
output_triton = add(x, y)
output_torch = x + y
print(f'Max diff: {torch.max(torch.abs(output_torch - output_triton))}')
# Should print 0.0

Max diff: 0.0


**Grid explained**: `triton.cdiv(n_elements, BLOCK_SIZE)` = ceiling division = number of blocks needed. The meta lambda pattern lets the grid depend on compile-time constants.

## Exercise: Fused Multiply-Add
Write a kernel that computes `output = x * 2 + 1` in a single pass.

In [21]:
@triton.jit
def fused_multiply_add_kernel(x_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)

    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    mask = offsets<n_elements

    x = tl.load(x_ptr+offsets, mask= mask)
    y = x*2+1
    tl.store(output_ptr+offsets, y, mask=mask )

def fused_multiply_add(x:torch.Tensor):
    output = torch.empty_like(x)
    assert x.is_cuda and output.is_cuda
    n_elements = output.numel()

    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    fused_multiply_add_kernel[grid](x, output, n_elements, BLOCK_SIZE=2048)
    return output




Benchmark against the two-op PyTorch version. Even for this trivial case, measure the difference.

In [22]:
import torch.utils.benchmark as benchmark

x = torch.randn(4096, 4096, device='cuda')

# Two separate ops 
# even though you did not explicitly write two kernels, PyTorch’s eager execution model treats mul and add as separate ops, each commonly backed by its own CUDA kernel.
#(2 HBM round-trips)
def two_ops(x):
    return (x * 2) + 1

output_triton = fused_multiply_add(x)
output_torch = two_ops(x)
print(f'Max diff: {torch.max(torch.abs(output_torch - output_triton))}')

# Measure
t = benchmark.Timer(stmt='two_ops(x)', globals={'two_ops': two_ops, 'x': x})
print(t.timeit(100))


t = benchmark.Timer(stmt='fused_multiply_add(x)', globals={'fused_multiply_add': fused_multiply_add, 'x': x})
print(t.timeit(100))


Max diff: 0.0
two_ops(x)
  292.09 us
  1 measurement, 100 runs , 1 thread
fused_multiply_add(x)
  147.80 us
  1 measurement, 100 runs , 1 thread


## What Controls Kernel Time

The execution time of a GPU kernel depends on:

1. **Hardware**: GPU model, number of SMs, memory bandwidth, cache size, clock speed, and available special compute units.
2. **Problem size**: total number of elements or operations the kernel must process.
3. **Grid size**: number of Triton program instances, similar to CUDA blocks / CTAs.
4. **Block size**: amount of work handled by each Triton program instance, for example `BLOCK_SIZE` elements.
5. **Memory access pattern**: contiguous and coalesced memory access is fast; scattered or repeated HBM access is slower.
6. **Compute pattern**: elementwise kernels are usually memory-bound, while matrix multiplication is usually compute-heavy.
7. **Occupancy and latency hiding**: enough independent work must be available so SMs can keep running while some warps wait on memory.
8. **Kernel launch overhead**: small workloads can be dominated by the cost of launching the kernel.
9. **CUDA stream / GPU queue**: kernels in the same stream run in order, and other GPU work can delay execution.

For the fused multiply-add example, `BLOCK_SIZE` should not be chosen as:

```python
BLOCK_SIZE = total_elements // num_sms
```

That creates only one large program per SM, which gives the scheduler less flexibility and can reduce occupancy. A smaller `BLOCK_SIZE`, such as `1024`, creates many more program instances. The SMs then pull new programs as they finish old ones, which helps hide memory latency and usually improves throughput.
